# Atividade de separatrizes: PIB per capita dos municípios do RN

Este é o **único notebook** do projeto. Ele faz o caminho completo, do download no IBGE até as respostas da atividade. Cada etapa começa com um **texto** (o que vamos fazer e por quê) e segue com **código Python** comentado para quem está começando.

Você **não precisa saber programar** para acompanhar: leia o texto e, no código, leia as linhas que começam com `#`. O Python ignora essas linhas; elas existem só para explicar.

## O que este notebook faz

1. Explica o conhecimento usado (separatriz, percentil, PIB per capita, DataFrame).
2. Importa as bibliotecas.
3. Baixa os dados oficiais do **IBGE Cidades@** (pelas APIs públicas por trás do portal).
4. Monta um **DataFrame** (tabela) com todos os municípios do Rio Grande do Norte.
5. **Exporta essa tabela para Excel (`.xlsx`)**, pronta para abrir no **Google Planilhas**.
6. Mostra todas as colunas.
7. Explica e calcula o **percentil 10**, também no Excel e no Google Planilhas.
8. Responde as perguntas da atividade, com tabelas (DataFrames) e gráficos.

## Objetivo da atividade

1. Obter o **PIB per capita** de **todos os municípios do Rio Grande do Norte**.
2. Criar o grupo com os **10% municípios de menor PIB per capita**.
   - **2.1** Calcular o **percentil 10** (a separatriz).
   - **2.2** Separar os municípios com PIB per capita **menor que** o percentil 10.
   - **2.3** Apresentar variáveis que possam explicar esses PIBs per capita.
3. **Descrever sumariamente** essa situação.

Portal de referência (página, não é API): [IBGE Cidades@ — Panorama do RN](https://cidades.ibge.gov.br/brasil/rn/panorama).

## Conhecimento que será aplicado

| Conceito | Em uma frase |
| --- | --- |
| **PIB per capita** | PIB do município dividido pela população. É produção média por habitante, **não** a renda de cada pessoa. |
| **Separatriz** | Valor que **corta** uma distribuição ordenada (aqui, o percentil 10). |
| **Percentil 10 (P10)** | Valor que deixa cerca de **10%** das observações **abaixo** dele. |
| **DataFrame** | Tabela do pandas: linhas = municípios, colunas = indicadores. |
| **API** | Endereço na internet que devolve dados (JSON), em vez de uma página bonita. |
| **Cidades@** | Portal do IBGE. Os números desta atividade saem das **APIs oficiais** que alimentam esse portal. |

## Como executar

1. **Run All** / **Ambiente de execução → Executar tudo**.
2. A coleta no IBGE precisa de **internet** (cerca de 1 minuto). O Excel gerado vai para `data/`, pronto para o Google Planilhas.
3. No **Google Colab**, envie **só este arquivo** `.ipynb`. Tudo que fala com o IBGE está nas células abaixo.


## Mini glossário

| Palavra | Significado aqui |
| --- | --- |
| **Notebook** | Este arquivo: mistura texto (explicação) e código (cálculo). |
| **Célula** | Um bloco. Células de texto explicam. Células de código calculam. |
| **Biblioteca** | Pacote pronto de funções. Exemplo: `pandas` trabalha com tabelas. |
| **Variável** | Uma caixinha com nome que guarda um valor. Exemplo: `percentil_10`. |
| **Função** | Um comando que recebe dados e devolve um resultado. Exemplo: `mean()` calcula a média. |
| **JSON** | Formato de texto que o IBGE envia; o Python transforma isso em tabela. |
| **UF 24** | Código do IBGE para o Rio Grande do Norte. |
| **N6[N3[24]]** | “Todos os municípios (N6) que estão no estado 24”. |

No código, tudo depois de `#` é só explicação.


# Passo 0. Importar as bibliotecas

Antes de baixar qualquer dado, carregamos as ferramentas. Cada biblioteca tem um papel:

| Biblioteca | Para que serve neste notebook |
| --- | --- |
| **pandas** (`pd`) | Cria e manipula **DataFrames** (tabelas). |
| **numpy** (`np`) | Calcula o **percentil** (separatriz). |
| **pathlib** | Monta caminhos de pastas e arquivos (`data/municipios_rn.xlsx`). |
| **plotly.express** (`px`) | Gráficos interativos (histograma, barras). |
| **openpyxl** | Grava o arquivo **Excel** usado no Google Planilhas. |
| **urllib** / **json** / **gzip** | Pedem os dados às APIs do IBGE e leem a resposta (JSON). |

A célula abaixo **instala** (se faltar) e **importa**. Ela ainda **não** fala com o IBGE.


In [ ]:
# Instala as bibliotecas. No Colab isso é necessário na primeira execução.
# O "-q" deixa a instalação quieta (menos texto na tela).
%pip install -q pandas numpy openpyxl plotly

# pandas: tabelas (DataFrame). Convenção: apelidar de pd.
import pandas as pd

# numpy: contas estatísticas. Aqui usamos np.percentile para o P10.
import numpy as np

# Path: caminho de arquivo de forma organizada (Windows, Mac, Colab).
from pathlib import Path

# json: o IBGE responde em JSON (texto organizado). gzip: a resposta pode vir compactada.
import gzip
import json
import time

# urllib: faz o pedido HTTP (GET) para as APIs do IBGE, sem biblioteca extra.
from urllib.error import HTTPError, URLError
from urllib.parse import quote
from urllib.request import Request, urlopen

# plotly.express: gráficos interativos (histograma, barras comparativas).
import plotly.express as px

# display: mostra mais de uma tabela na mesma célula (Jupyter / Colab).
from IPython.display import display

# Números na tela no padrão brasileiro: 13668.37 aparece como 13.668,37
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 40)
pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."),
)

# Pasta onde vamos gravar o Excel (a pasta atual do notebook).
raiz = Path.cwd()


def br(valor, casas=2):
    """Escreve um número no formato brasileiro, só para leitura.

    Exemplo: 13668.37 vira o texto "13.668,37".
    Isso NÃO altera o cálculo, só o que aparece na tela.
    """
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return "-"
    texto = f"{float(valor):,.{casas}f}"
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")


print("Bibliotecas carregadas.")
print("Pasta de trabalho:", raiz)
print("Exemplo de formatação (não é dado do IBGE):", br(13668.37))


# Passo 1. Quais dados precisamos, de onde vêm e para que servem

A atividade pede o PIB per capita de **todos** os municípios do RN e, depois, variáveis que ajudem a **entender** os menores valores. Por isso não basta uma coluna só.

## O portal Cidades@ não é uma API

O endereço [cidades.ibge.gov.br/brasil/rn/panorama](https://cidades.ibge.gov.br/brasil/rn/panorama) é uma **página**. Por trás dela, o IBGE publica **APIs oficiais** (serviços que devolvem JSON). Este notebook consulta essas APIs **direto nas células abaixo**. **Não raspa HTML** e não inventa números.

Há dois códigos de município:

- **7 dígitos** na API de Localidades e na de Agregados (exemplo: `2400109` = Acari).
- **6 dígitos** na API de Pesquisas / Cidades@ (exemplo: `240010`).

O notebook guarda o código de 7 dígitos e usa os 6 primeiros para cruzar as fontes.

O recorte `N6[N3[24]]` significa: todos os municípios (nível N6) dentro da UF **24** (Rio Grande do Norte).

## Tabela dos dados que vamos baixar

| Dado | Fonte | Ano | Por que entra |
| --- | --- | --- | --- |
| Lista dos 167 municípios | API de Localidades | vigente | Universo da atividade (pergunta 1) |
| PIB per capita (R$) | Cidades@, pesquisa 38, indicador **47001** | 2023 | Variável da separatriz |
| PIB total (R$ mil) | Cidades@, pesquisa 38, indicador **46997** | 2023 | Tamanho da economia |
| VAB agro, indústria, serviços, adm. pública | Cidades@, pesquisa 38, indicadores 47006–47009 | 2021 | Estrutura produtiva (pergunta 2.3) |
| População, área, densidade | SIDRA tabela 4714 | Censo 2022 | Porte e dispersão |
| Alfabetização 15+ | SIDRA tabela 9543 | Censo 2022 | Indicador educacional |
| IDHM, escolarização, salário, ocupação | Pesquisas 10111 e 10058 | último disponível | Desenvolvimento e mercado de trabalho |

Observações importantes:

- O PIB per capita do Cidades@ é o indicador **47001**, não a variável SIDRA `543` (impostos).
- Para 2022 e 2023 o IBGE divulgou o PIB, mas a **composição setorial completa** mais recente é **2021**. Por isso o notebook mistura anos oficiais diferentes, sempre citados.

As funções da próxima célula são as que **pedem** esses dados. Cada uma está explicada no bloco de texto abaixo, antes do código.


In [ ]:
## 1.0 Funções que este notebook usa para falar com o IBGE

Uma **função** é um pedaço de código com nome. Você chama pelo nome, ela executa os passos de dentro e devolve um resultado. Criamos várias funções pequenas para não repetir o mesmo pedido à internet dezenas de vezes.

Elas **não baixam ainda** a tabela final: só ficam **prontas**. A coleta em si acontece nas células seguintes.

### `br(valor, casas=2)`

Já foi definida no passo 0. **Não calcula estatística.** Só escreve um número no formato brasileiro (`13668.37` vira `"13.668,37"`) para a leitura na tela. O cálculo continua com ponto decimal do Python.

### `para_numero(valor)`

O IBGE às vezes manda o número como texto (`"13668.37"`), às vezes manda um código de dado faltando (`"-"`, `"..."`, `"X"`). Esta função tenta virar o valor em `float`. Se não der, devolve `None` (vazio), para a conta do percentil não quebrar.

### `baixar_json(url)`

É o “telefone” do notebook. Recebe um **endereço** (URL da API), faz um GET HTTP, espera a resposta, descompacta se vier em gzip e transforma o JSON em estruturas do Python (`dict` e `list`).

Se a API falhar (rede, timeout), ela **tenta de novo** até 4 vezes, esperando um pouco entre as tentativas. Se as 4 falharem, o notebook para com uma mensagem clara. Sem esta função, cada indicador precisaria copiar o mesmo bloco de internet.

### `listar_municipios_rn()`

Pede à **API de Localidades** a lista oficial dos municípios da UF 24. Devolve uma **lista de dicionários**, um por município, com:

- código de 7 dígitos e os 6 primeiros (para cruzar com o Cidades@);
- nome;
- microrregião, mesorregião, região imediata e intermediária.

URL:

```
https://servicodados.ibge.gov.br/api/v1/localidades/estados/24/municipios?orderBy=nome
```

### `consultar_cidades(pesquisa, indicador, periodo)`

Pede um indicador da **API de Pesquisas**, a mesma base do portal Cidades@.

- `pesquisa` — número da pesquisa (38 = PIB dos municípios).
- `indicador` — o dado (47001 = PIB per capita).
- `periodo` — o ano (`"2023"`), quando a pesquisa tem série anual.

O recorte territorial é sempre `N6[N3[24]]` (todos os municípios do RN). A função devolve um dicionário:

```text
{ código de 6 dígitos → { ano → valor } }
```

Exemplo de chave: `"240010"` (Acari). Assim dá para juntar com a lista de municípios.

### `consultar_cidades_opcional(...)`

Igual à anterior, mas **não interrompe** o notebook se aquele indicador estiver fora do ar. IDHM, salário e ocupação são complementares: se falharem, a coluna fica vazia e o restante da atividade segue.

### `consultar_sidra(agregado, variaveis, periodo, classificacao)`

Pede dados da **API de Agregados** (SIDRA), usada no Censo.

- `agregado` — número da tabela (4714 = população, área, densidade).
- `variaveis` — códigos SIDRA, separados por `|` quando há vários.
- `periodo` — ano (`"2022"`).
- `classificacao` — filtros (sexo, cor, idade). Na alfabetização usamos os totais, para não quebrar a taxa em fatias.

Devolve:

```text
{ código de 7 dígitos → { id da variável → valor } }
```

Aqui a chave tem **7 dígitos**, igual à API de Localidades.

### `valor_do_ano(serie, ano)` e `ultimo_valor(serie)`

Cada indicador chega como uma **série** (vários anos). `valor_do_ano` pega o ano pedido (2023 no PIB, 2021 no VAB). `ultimo_valor` pega o ano mais recente que não esteja vazio — útil no IDHM, que não é divulgado todo ano.

### `soma_segura(lista)` e `participacao(parte, total)`

`soma_segura` soma só os números que existem (ignora `None`). `participacao` calcula `100 × parte / total`: o peso de um setor no valor adicionado do município. Se faltar dado, devolve `None`.

### `montar_dataframe_rn()`

É a função **orquestradora**. Na ordem:

1. lista os 167 municípios;
2. baixa PIB, VAB, Censo, alfabetização e indicadores sociais;
3. para **cada município**, junta as peças pelos códigos 6 e 7;
4. calcula as participações setoriais;
5. devolve **um DataFrame**, ainda **sem** o percentil 10 (isso vem no passo 4, à vista).

A célula de código seguinte **define** essas funções. As células depois **chamam** `listar_municipios_rn()` e `montar_dataframe_rn()`.


In [ ]:
# Endereços oficiais das APIs do IBGE (as mesmas bases do Cidades@)
IBGE_PESQUISAS = "https://servicodados.ibge.gov.br/api/v1/pesquisas"
IBGE_LOCALIDADES = "https://servicodados.ibge.gov.br/api/v1/localidades"
IBGE_AGREGADOS = "https://servicodados.ibge.gov.br/api/v3/agregados"

CODIGO_RN = "24"  # código da UF Rio Grande do Norte
MUNICIPIOS_DO_RN = "N6[N3[24]]"  # todos os municípios dentro do estado 24

ANO_PIB = "2023"
ANO_VAB = "2021"
ANO_CENSO = "2022"


def para_numero(valor):
    """Converte a resposta do IBGE em número, ou None se o dado não existir."""
    if valor is None or valor in ("", "-", "...", "X"):
        return None
    try:
        return float(str(valor).replace(",", "."))
    except (TypeError, ValueError):
        return None


def baixar_json(url, tentativas=4):
    """GET em uma URL do IBGE. Devolve o JSON já virado em dict/list do Python."""
    ultimo_erro = None
    for i in range(tentativas):
        try:
            pedido = Request(
                url,
                headers={
                    "User-Agent": "PIB-municipios-RN (atividade academica)",
                    "Accept": "application/json",
                    "Accept-Encoding": "gzip",
                },
            )
            with urlopen(pedido, timeout=120) as resposta:
                bruto = resposta.read()
                # Algumas respostas vêm compactadas (gzip)
                if resposta.headers.get("Content-Encoding") == "gzip" or bruto[:2] == b"\x1f\x8b":
                    bruto = gzip.decompress(bruto)
                return json.loads(bruto.decode("utf-8"))
        except (HTTPError, URLError, TimeoutError, json.JSONDecodeError) as erro:
            ultimo_erro = erro
            time.sleep(1.5 * (i + 1))  # espera um pouco e tenta de novo
    raise RuntimeError(f"Falha ao consultar {url}: {ultimo_erro}")


def listar_municipios_rn():
    """Devolve a lista dos 167 municípios do RN (nome, código e recortes)."""
    url = f"{IBGE_LOCALIDADES}/estados/{CODIGO_RN}/municipios?orderBy=nome"
    bruto = baixar_json(url)
    municipios = []
    for item in bruto:
        micro = item.get("microrregiao") or {}
        meso = micro.get("mesorregiao") or {}
        imediata = item.get("regiao-imediata") or {}
        intermediaria = imediata.get("regiao-intermediaria") or {}
        codigo = str(item["id"])
        municipios.append(
            {
                "codigo_municipio": codigo,
                "codigo_municipio_6": codigo[:6],
                "municipio": item["nome"],
                "uf": "RN",
                "microrregiao": micro.get("nome"),
                "mesorregiao": meso.get("nome"),
                "regiao_imediata": imediata.get("nome"),
                "regiao_intermediaria": intermediaria.get("nome"),
            }
        )
    return municipios


def consultar_cidades(pesquisa, indicador, periodo=None, tentativas=4):
    """Consulta a API de Pesquisas (Cidades@). Devolve {codigo_6: {ano: valor}}."""
    local = quote(MUNICIPIOS_DO_RN, safe="[]")
    if periodo:
        url = (
            f"{IBGE_PESQUISAS}/{pesquisa}/periodos/{periodo}"
            f"/indicadores/{indicador}/resultados/{local}"
        )
    else:
        url = f"{IBGE_PESQUISAS}/{pesquisa}/indicadores/{indicador}/resultados/{local}"
    dados = baixar_json(url, tentativas=tentativas)
    saida = {}
    for bloco in dados:
        for registro in bloco.get("res", []):
            codigo = str(registro["localidade"])
            serie = {}
            for ano, valor in (registro.get("res") or {}).items():
                serie[str(ano)] = para_numero(valor)
            saida[codigo] = serie
    return saida


def consultar_cidades_opcional(pesquisa, indicador, periodo=None):
    """Igual a consultar_cidades, mas devolve um dicionário vazio se aquele indicador falhar."""
    try:
        return consultar_cidades(pesquisa, indicador, periodo, tentativas=2)
    except RuntimeError as erro:
        print(f"Indicador opcional {pesquisa}/{indicador} indisponível:", erro)
        return {}


def consultar_sidra(agregado, variaveis, periodo, classificacao=None):
    """Consulta a API de Agregados (SIDRA). Devolve {codigo_7: {id_variavel: valor}}."""
    local = quote(MUNICIPIOS_DO_RN, safe="[]")
    url = (
        f"{IBGE_AGREGADOS}/{agregado}/periodos/{periodo}"
        f"/variaveis/{variaveis}?localidades={local}"
    )
    if classificacao:
        url += f"&classificacao={quote(classificacao, safe='[]|,')}"
    dados = baixar_json(url)
    saida = {}
    for variavel in dados:
        vid = str(variavel["id"])
        for resultado in variavel.get("resultados", []):
            for serie in resultado.get("series", []):
                codigo = str(serie["localidade"]["id"])
                valores = serie.get("serie") or {}
                valor = next(iter(valores.values()), None)
                saida.setdefault(codigo, {})[vid] = para_numero(valor)
    return saida


def valor_do_ano(serie, ano):
    """Pega o valor de um ano específico dentro da série {ano: valor}."""
    if not serie:
        return None
    return serie.get(ano)


def ultimo_valor(serie):
    """Pega o valor mais recente que não esteja vazio."""
    if not serie:
        return None
    for ano in sorted(serie.keys(), reverse=True):
        if serie[ano] is not None:
            return serie[ano]
    return None


def soma_segura(valores):
    """Soma ignorando None. Se não sobrar nenhum número, devolve None."""
    nums = [v for v in valores if v is not None]
    if not nums:
        return None
    return float(sum(nums))


def participacao(parte, total):
    """Peso percentual: 100 * (parte / total). None se faltar dado."""
    if parte is None or not total:
        return None
    return 100.0 * parte / total


def montar_dataframe_rn():
    """Baixa todos os indicadores e devolve um DataFrame (ainda sem percentil)."""
    municipios = pd.DataFrame(listar_municipios_rn())

    print("Buscando PIB per capita e PIB total (Cidades@, 2023)...")
    pib_pc = consultar_cidades(38, 47001, periodo=ANO_PIB)
    pib_total = consultar_cidades(38, 46997, periodo=ANO_PIB)

    print("Buscando valor adicionado por setor (Cidades@, 2021)...")
    vab_agro = consultar_cidades(38, 47006, periodo=ANO_VAB)
    vab_industria = consultar_cidades(38, 47007, periodo=ANO_VAB)
    vab_servicos = consultar_cidades(38, 47008, periodo=ANO_VAB)
    vab_adm = consultar_cidades(38, 47009, periodo=ANO_VAB)

    print("Buscando Censo 2022 (população, área, densidade e alfabetização)...")
    censo = consultar_sidra(4714, "93|6318|614", ANO_CENSO)
    alfabetizacao = consultar_sidra(
        9543,
        "2513",
        ANO_CENSO,
        classificacao="2[6794]|86[95251]|287[100362]",
    )

    print("Buscando indicadores sociais complementares (IDHM, escolarização, salário, ocupação)...")
    idhm = consultar_cidades_opcional(10111, 329756)
    escolarizacao = consultar_cidades_opcional(10058, 60045)
    salario_medio = consultar_cidades_opcional(10058, 60038)
    populacao_ocupada = consultar_cidades_opcional(10058, 60036)

    linhas = []
    for _, mun in municipios.iterrows():
        codigo7 = mun["codigo_municipio"]
        codigo6 = mun["codigo_municipio_6"]
        demo = censo.get(codigo7, {})
        alfa = alfabetizacao.get(codigo7, {})
        agro = valor_do_ano(vab_agro.get(codigo6), ANO_VAB)
        industria = valor_do_ano(vab_industria.get(codigo6), ANO_VAB)
        servicos = valor_do_ano(vab_servicos.get(codigo6), ANO_VAB)
        adm = valor_do_ano(vab_adm.get(codigo6), ANO_VAB)
        vab_total = soma_segura([agro, industria, servicos, adm])
        linhas.append(
            {
                "codigo_municipio": codigo7,
                "municipio": mun["municipio"],
                "uf": mun["uf"],
                "mesorregiao": mun["mesorregiao"],
                "microrregiao": mun["microrregiao"],
                "regiao_imediata": mun["regiao_imediata"],
                "regiao_intermediaria": mun["regiao_intermediaria"],
                "ano_pib": int(ANO_PIB),
                "pib_mil_reais": valor_do_ano(pib_total.get(codigo6), ANO_PIB),
                "pib_per_capita": valor_do_ano(pib_pc.get(codigo6), ANO_PIB),
                "populacao_censo_2022": demo.get("93"),
                "area_km2": demo.get("6318"),
                "densidade_demografica": demo.get("614"),
                "ano_vab": int(ANO_VAB),
                "vab_agropecuaria_mil_reais": agro,
                "vab_industria_mil_reais": industria,
                "vab_servicos_mil_reais": servicos,
                "vab_administracao_publica_mil_reais": adm,
                "participacao_agropecuaria_pct": participacao(agro, vab_total),
                "participacao_industria_pct": participacao(industria, vab_total),
                "participacao_servicos_pct": participacao(servicos, vab_total),
                "participacao_adm_publica_pct": participacao(adm, vab_total),
                "taxa_alfabetizacao_15_mais_pct": alfa.get("2513"),
                "idhm": ultimo_valor(idhm.get(codigo6)),
                "taxa_escolarizacao_6_a_14_pct": ultimo_valor(escolarizacao.get(codigo6)),
                "salario_medio_mensal": ultimo_valor(salario_medio.get(codigo6)),
                "populacao_ocupada": ultimo_valor(populacao_ocupada.get(codigo6)),
            }
        )

    tabela = pd.DataFrame(linhas)
    return tabela.sort_values("pib_per_capita", ascending=True, ignore_index=True)


print("Funções definidas. Elas ainda não baixaram os dados — isso acontece nas células seguintes.")
print("UF:", CODIGO_RN, "| recorte:", MUNICIPIOS_DO_RN)


## 1.1 Lista dos municípios do RN (primeiro DataFrame)

Começamos pelo cadastro territorial. A API de Localidades devolve os **167 municípios** do estado 24, com nome, código IBGE, mesorregião e microrregião.

Endereço usado:

```
https://servicodados.ibge.gov.br/api/v1/localidades/estados/24/municipios?orderBy=nome
```

O resultado entra em um DataFrame chamado `municipios`. Cada **linha** é um município.


In [ ]:
# DataFrame = tabela. Cada linha será um município do RN.
municipios = pd.DataFrame(listar_municipios_rn())

print("Quantidade de municípios do RN:", len(municipios))
print("Colunas deste primeiro DataFrame:", list(municipios.columns))
print()
print("Amostra (primeiras linhas, ordem alfabética do IBGE):")
municipios.head()


## 1.2 Baixar os indicadores e montar o DataFrame da atividade

A função `montar_dataframe_rn()` (definida acima) faz, nesta ordem:

1. PIB per capita e PIB total (2023) — Cidades@, pesquisa 38.
2. Valor adicionado por setor (2021) — mesma pesquisa, outros indicadores.
3. População, área, densidade e alfabetização — Censo 2022 (SIDRA).
4. IDHM e indicadores sociais — pesquisas complementares.
5. Cruza os códigos (6 e 7 dígitos) e devolve **um DataFrame só**, ainda **sem** o percentil.

O P10 será calculado mais adiante, à vista, com NumPy.

Espere cerca de **um minuto**. Precisa de internet.


In [ ]:
print("Iniciando a coleta nas APIs oficiais do IBGE (cerca de 1 minuto)...")
print("Portal de referência: https://cidades.ibge.gov.br/brasil/rn/panorama")
print("O percentil ainda NÃO é calculado aqui; isso vem no passo 4.")
print()

# df é o DataFrame principal da atividade: uma linha por município.
# montar_dataframe_rn() está definida neste notebook, não em outro arquivo.
df = montar_dataframe_rn()

print()
print("Coleta concluída.")
print("O que shape significa: (número de linhas, número de colunas).")
print("Linhas (municípios):", df.shape[0])
print("Colunas (variáveis):", df.shape[1])
print()
print("Primeiras linhas, já ordenadas do menor PIB per capita para o maior:")
df.head()


# Passo 2. Exportar o DataFrame para Excel (Google Planilhas)

O arquivo `.xlsx` é o formato que o **Google Planilhas** abre sem conversão estranha de CSV (acentos e casas decimais).

Como usar no Google Planilhas:

1. Baixe `data/municipios_rn.xlsx` (no Colab: pasta à esquerda da tela).
2. Acesse [Google Planilhas](https://sheets.google.com) → **Arquivo → Importar** (ou envie o arquivo para o Drive e abra com Planilhas).
3. A aba `Todos_municipios` tem uma linha de cabeçalho e 167 municípios.

A célula abaixo cria a pasta `data/` se ela ainda não existir e grava o Excel. Depois, quando calculamos o P10, atualizamos o mesmo arquivo com mais abas.


In [ ]:
# Cria a pasta data/ se ainda não existir. exist_ok=True evita erro se ela já existir.
pasta_dados = raiz / "data"
pasta_dados.mkdir(exist_ok=True)

# Caminho do Excel que você vai mandar para o Google Planilhas
arquivo_xlsx = pasta_dados / "municipios_rn.xlsx"

# index=False: não grava a coluna 0, 1, 2... do pandas (só os dados do IBGE)
with pd.ExcelWriter(arquivo_xlsx, engine="openpyxl") as excel:
    df.to_excel(excel, sheet_name="Todos_municipios", index=False)

print("Excel salvo em:", arquivo_xlsx)
print("Abas neste momento: Todos_municipios")
print("No Colab: use a pasta à esquerda para baixar o arquivo e abrir no Google Planilhas.")


# Passo 3. Todas as colunas do DataFrame

Cada coluna é uma variável. A tabela abaixo é ela mesma um DataFrame: nome da coluna, tipo e um exemplo (primeira linha). Use essa lista como dicionário ao abrir o Excel no Google Planilhas.


In [ ]:
# Montamos um DataFrame só para documentar as colunas (não altera df).
dicionario_colunas = pd.DataFrame(
    {
        "coluna": df.columns,
        "tipo": [str(t) for t in df.dtypes],
        "exemplo_primeira_linha": [df.iloc[0][c] for c in df.columns],
    }
)

print("Quantidade de colunas:", len(df.columns))
print("Lista completa:", list(df.columns))
print()
dicionario_colunas


Significado das colunas principais:

| Coluna | O que é |
| --- | --- |
| `codigo_municipio` | Código IBGE de 7 dígitos |
| `municipio` | Nome da cidade |
| `mesorregiao` / `microrregiao` | Recortes territoriais do IBGE |
| `pib_per_capita` | PIB por habitante em 2023, em reais |
| `pib_mil_reais` | PIB total de 2023, em mil reais |
| `populacao_censo_2022` | Habitantes no Censo 2022 |
| `area_km2` / `densidade_demografica` | Área e habitantes por km² |
| `participacao_*_pct` | Peso de cada setor no valor adicionado de **2021** |
| `taxa_alfabetizacao_15_mais_pct` | % de pessoas com 15 anos ou mais que sabem ler e escrever |
| `idhm` | Índice de Desenvolvimento Humano Municipal |


# Passo 4. O que é percentil e como se calcula

## Ideia

Imagine todos os municípios em **fila**, do menor PIB per capita para o maior. O **percentil 10** é o valor que fica na posição “10% dessa fila”.

- Cerca de **10%** dos municípios ficam **abaixo** desse valor.
- Cerca de **90%** ficam **iguais ou acima**.

Isso é uma **separatriz**: um corte estatístico, não um “top 17” escolhido no olho.

Dez por cento de 167 municípios = 16,7. Por isso o grupo da pergunta 2.2 deve ter **cerca de 17** municípios.

## Método (o mesmo do Excel e do NumPy)

Usamos o método **inclusivo** (interpolação linear), equivalente a:

| Ferramenta | Fórmula |
| --- | --- |
| **Python / NumPy** | `np.percentile(valores, 10)` — método padrão `linear` |
| **Excel** | `=PERCENTIL.INC(intervalo; 0,1)` |
| **Google Planilhas** | `=PERCENTILE(intervalo; 0,1)` |

Passo a passo numérico (reproduzível à mão ou na planilha):

1. Liste só a coluna `pib_per_capita` e **ordene** do menor para o maior. Chame o tamanho da lista de **n**.
2. Calcule o **índice** (começando do zero, como no Python):

   `i = (n - 1) × 0,10`

3. Se **i** não for inteiro, pegue o município na posição `chão(i)` e o da posição `teto(i)` e **interpole**:

   `P10 = valor_chão + (i - chão(i)) × (valor_teto - valor_chão)`

4. Arredonde para **2 casas** (reais e centavos).

5. A regra da atividade (pergunta 2.2) é: entra no grupo se **PIB per capita &lt; P10** (estritamente menor).

### No Google Planilhas / Excel, depois de importar o xlsx

Suponha que `pib_per_capita` esteja na coluna **J**, linhas 2 a 168 (cabeçalho na linha 1):

```
=PERCENTILE(J2:J168; 0,1)          → Google Planilhas (ou PERCENTIL.INC no Excel)
=ARRED(PERCENTILE(J2:J168; 0,1); 2)
```

Para marcar o grupo (coluna auxiliar):

```
=J2 < $P$1
```

onde `$P$1` é a célula em que você colocou o P10.

A célula seguinte faz **exatamente** esses passos no Python e confere com `np.percentile`.


In [ ]:
# 1) Pegamos só o PIB per capita, sem células vazias, e ordenamos do menor para o maior.
valores_ordenados = (
    df["pib_per_capita"]
    .dropna()
    .astype(float)
    .sort_values()
    .reset_index(drop=True)  # índice 0, 1, 2... na ordem crescente
)

n = len(valores_ordenados)
p = 10  # percentil 10 → 10%
fracao_percentil = p / 100  # 0,10

# 2) Índice no método linear / PERCENTIL.INC: (n - 1) * 0,10
indice = (n - 1) * fracao_percentil
i_chao = int(np.floor(indice))  # posição imediatamente abaixo (índice começando em 0)
i_teto = int(np.ceil(indice))   # posição imediatamente acima
parte_fracionaria = indice - i_chao

x_chao = float(valores_ordenados.iloc[i_chao])
x_teto = float(valores_ordenados.iloc[i_teto])

# 3) Interpolação linear entre os dois vizinhos
p10_passo_a_passo = x_chao + parte_fracionaria * (x_teto - x_chao)

# 4) O NumPy faz os passos 1–3 de uma vez (mesmo método)
p10_numpy = float(np.percentile(valores_ordenados, p))

# 5) Arredondamos para 2 casas: PIB per capita está em reais e centavos
percentil_10 = round(p10_numpy, 2)

# DataFrame de conferência: você pode copiar esses números para o Excel e repetir a conta
conferencia_p10 = pd.DataFrame(
    {
        "etapa": [
            "n (quantos municípios)",
            "percentil pedido",
            "índice i = (n-1)*0,10",
            "posição chão (índice 0)",
            "posição teto (índice 0)",
            "PIB na posição chão (R$)",
            "PIB na posição teto (R$)",
            "P10 interpolado (R$)",
            "P10 pelo NumPy (R$)",
            "P10 arredondado (R$)",
        ],
        "valor": [
            n,
            p,
            indice,
            i_chao,
            i_teto,
            x_chao,
            x_teto,
            p10_passo_a_passo,
            p10_numpy,
            percentil_10,
        ],
    }
)

print("Os dois caminhos (passo a passo e NumPy) devem coincidir.")
print("P10 arredondado = R$", br(percentil_10))
print()
print("Fórmula Excel / Google Planilhas (depois de importar o xlsx):")
print("  =PERCENTILE(coluna_do_pib; 0,1)")
print()
conferencia_p10


O gráfico abaixo mostra a **distribuição** do PIB per capita. A linha vermelha é o percentil 10: à esquerda dela ficam os municípios da pergunta 2.2.


In [ ]:
# Histograma: cada barra conta quantos municípios caem naquela faixa de PIB per capita.
fig_hist = px.histogram(
    df,
    x="pib_per_capita",
    nbins=25,
    title="Distribuição do PIB per capita municipal no RN (2023)",
    labels={"pib_per_capita": "PIB per capita (R$)", "count": "Número de municípios"},
)
# Linha vertical no P10 (separatriz)
fig_hist.add_vline(
    x=percentil_10,
    line_dash="dash",
    line_color="red",
    annotation_text=f"P10 = R$ {br(percentil_10)}",
    annotation_position="top right",
)
fig_hist.update_layout(bargap=0.05)
fig_hist.show()


---
# Pergunta 1. PIB per capita de todos os municípios do RN

**O que a pergunta pede:** a lista completa, não um recorte.

**O que vamos fazer:** criar um DataFrame só com identificação territorial e as colunas de PIB, ordenado do menor para o maior PIB per capita.

**Por que isso importa:** o percentil 10 é uma posição **dentro da distribuição de todos** os 167 municípios. Sem a lista completa o P10 não tem sentido.

**Fonte:** IBGE Cidades@, pesquisa 38, indicador 47001, ano **2023**.


In [ ]:
# Copia só as colunas da pergunta 1 (não apaga o DataFrame original df)
pib_todos = df[
    ["codigo_municipio", "municipio", "mesorregiao", "microrregiao", "pib_per_capita", "pib_mil_reais"]
].copy()

# ascending=True = do menor PIB per capita para o maior
pib_todos = pib_todos.sort_values("pib_per_capita", ascending=True)

# ranking 1 = município com o menor PIB per capita do estado
pib_todos["ranking"] = range(1, len(pib_todos) + 1)

# Resumo estatístico em um DataFrame (não em texto solto)
resumo_pergunta_1 = pd.DataFrame(
    {
        "estatística": [
            "Quantidade de municípios",
            "Ano do PIB",
            "Menor PIB per capita (R$)",
            "Maior PIB per capita (R$)",
            "Média (R$)",
            "Mediana (R$)",
        ],
        "valor": [
            len(pib_todos),
            2023,
            pib_todos["pib_per_capita"].min(),
            pib_todos["pib_per_capita"].max(),
            pib_todos["pib_per_capita"].mean(),
            pib_todos["pib_per_capita"].median(),
        ],
    }
)

print("Resposta da pergunta 1 — resumo:")
display(resumo_pergunta_1)

print()
print("A mediana é o valor do meio da fila.")
print("Se a média fica bem acima da mediana, poucos municípios muito ricos puxam a média.")
print()
print("Lista completa (role a tabela). Ano: 2023.")
pib_todos


Os extremos ajudam a ver a desigualdade. Abaixo: DataFrame dos 5 menores e DataFrame dos 5 maiores, e um gráfico de barras dos 15 menores.


In [ ]:
menores_5 = pib_todos.head(5).copy()
maiores_5 = pib_todos.tail(5).copy()

print("5 municípios com MENOR PIB per capita")
display(menores_5)

print("5 municípios com MAIOR PIB per capita")
display(maiores_5)

# Gráfico: 15 menores PIB per capita (já estão no topo de pib_todos)
quinze_menores = pib_todos.head(15)
fig_q1 = px.bar(
    quinze_menores,
    x="pib_per_capita",
    y="municipio",
    orientation="h",
    title="15 menores PIB per capita municipais no RN (2023)",
    labels={"pib_per_capita": "PIB per capita (R$)", "municipio": "Município"},
)
fig_q1.update_layout(yaxis={"categoryorder": "total ascending"})
fig_q1.show()


---
# Pergunta 2.1 Calcular o percentil 10

**O que a pergunta pede:** o valor da separatriz (P10) do PIB per capita.

**O que já fizemos:** no passo 4 o P10 foi calculado de duas formas (interpolação à mão e `np.percentile`) e conferido.

**Resposta:** o número abaixo é o percentil 10. Na coleta de referência deste projeto ele ficou em torno de **R$ 13.668,37**. Se o IBGE revisar a série, vale o valor impresso agora.


In [ ]:
# DataFrame-resposta da pergunta 2.1 (uma linha só)
resposta_2_1 = pd.DataFrame(
    {
        "indicador": ["Percentil 10 do PIB per capita"],
        "ano": [2023],
        "n_municipios": [n],
        "valor_R$": [percentil_10],
        "regra_do_grupo": ["PIB per capita < P10"],
        "excel_google_sheets": ["=PERCENTILE(coluna_pib; 0,1)"],
    }
)

print("Cerca de 10% dos municípios do RN têm PIB per capita menor que R$", br(percentil_10) + ".")
resposta_2_1


---
# Pergunta 2.2 Municípios com PIB per capita menor que o percentil 10

**Regra (ao pé da letra):**

```text
entra no grupo se  PIB per capita  <  percentil 10
```

O símbolo `<` significa **estritamente menor**. Quem tiver exatamente o valor do P10 **não entra**.

Criamos uma coluna lógica (`True`/`False`) e filtramos. O resultado é um **novo DataFrame** (`grupo_p10`).


In [ ]:
# True = o município entra no grupo da pergunta 2.2
df["abaixo_do_p10"] = df["pib_per_capita"] < percentil_10
df["percentil_10_pib_per_capita"] = percentil_10

# Os colchetes filtram: ficam só as linhas em que a condição é True
grupo_p10 = df[df["abaixo_do_p10"]].copy()
grupo_p10 = grupo_p10.sort_values("pib_per_capita")

# DataFrame só com as colunas da pergunta 2.2
grupo_p10_resposta = grupo_p10[
    [
        "municipio",
        "mesorregiao",
        "microrregiao",
        "pib_per_capita",
        "populacao_censo_2022",
        "densidade_demografica",
    ]
].copy()
grupo_p10_resposta["ranking_no_grupo"] = range(1, len(grupo_p10_resposta) + 1)

print("Regra aplicada: pib_per_capita <", br(percentil_10))
print("Municípios no grupo:", len(grupo_p10_resposta))
print(
    "Isso equivale a",
    br(100 * len(grupo_p10_resposta) / len(df), 1),
    "% dos municípios do RN.",
)
print()
grupo_p10_resposta


In [ ]:
# Gráfico do grupo: barras horizontais do menor para o maior PIB per capita
fig_grupo = px.bar(
    grupo_p10_resposta,
    x="pib_per_capita",
    y="municipio",
    orientation="h",
    title=f"Municípios com PIB per capita menor que o P10 (R$ {br(percentil_10)})",
    labels={"pib_per_capita": "PIB per capita 2023 (R$)", "municipio": "Município"},
    hover_data=["mesorregiao", "populacao_censo_2022"],
)
fig_grupo.update_layout(yaxis={"categoryorder": "total ascending"}, height=520)
fig_grupo.add_vline(x=percentil_10, line_dash="dash", line_color="red")
fig_grupo.show()


---
# Pergunta 2.3 Variáveis que podem explicar esses PIBs per capita

O PIB per capita é uma **razão**: produção do município ÷ população. Um valor baixo pode aparecer quando:

- a economia é **pequena** e pouco diversificada;
- quase tudo que se produz vem da **administração pública**;
- há **pouca indústria**;
- o município é **pequeno e pouco denso**;
- os indicadores de **educação e IDHM** são mais fracos.

Por isso comparamos o **grupo P10** com os **demais municípios do RN**. Se uma diferença aparece com clareza nos dois grupos, ela **ajuda a explicar** o recorte (não prova uma única causa).

| Variável | Por que entra |
| --- | --- |
| População e densidade | Municípios menores tendem a ter menos atividade econômica |
| Área | Porte físico, junto com a densidade |
| % administração pública no VAB | Dependência de salários e serviços públicos |
| % indústria e serviços | Sinal de diversificação produtiva |
| % agropecuária | Peso do setor primário |
| Alfabetização e escolarização | Escolaridade ligada à produtividade |
| IDHM | Síntese de renda, educação e longevidade |
| Salário médio e população ocupada | Mercado de trabalho formal |

O VAB setorial é de **2021**; o PIB é de **2023**; o Censo é de **2022**.


In [ ]:
# Os "demais" são todos os que NÃO estão abaixo do P10
# O símbolo ~ inverte verdadeiro/falso: True vira False e vice-versa
demais = df[~df["abaixo_do_p10"]].copy()

print("Municípios no grupo P10:", len(grupo_p10))
print("Demais municípios do RN:", len(demais))

# Cada item: (nome da coluna, rótulo para leitura, casas decimais)
variaveis = [
    ("pib_per_capita", "PIB per capita (R$)", 2),
    ("pib_mil_reais", "PIB total (R$ mil)", 2),
    ("populacao_censo_2022", "População (Censo 2022)", 0),
    ("densidade_demografica", "Densidade (hab/km²)", 1),
    ("area_km2", "Área (km²)", 1),
    ("participacao_agropecuaria_pct", "Participação da agropecuária no VAB (%)", 1),
    ("participacao_industria_pct", "Participação da indústria no VAB (%)", 1),
    ("participacao_servicos_pct", "Participação dos serviços no VAB (%)", 1),
    ("participacao_adm_publica_pct", "Participação da administração pública no VAB (%)", 1),
    ("taxa_alfabetizacao_15_mais_pct", "Alfabetização 15 anos ou mais (%)", 1),
    ("idhm", "IDHM", 3),
    ("taxa_escolarizacao_6_a_14_pct", "Escolarização 6 a 14 anos (%)", 1),
    ("salario_medio_mensal", "Salário médio (salários mínimos)", 2),
    ("populacao_ocupada", "População ocupada (%)", 1),
]

linhas_comparativo = []
for coluna, rotulo, casas in variaveis:
    # mean() = média aritmética; median() = valor do meio da fila
    media_grupo = grupo_p10[coluna].mean()
    media_demais = demais[coluna].mean()
    mediana_grupo = grupo_p10[coluna].median()
    mediana_demais = demais[coluna].median()
    linhas_comparativo.append(
        {
            "Variável": rotulo,
            "Média grupo P10": media_grupo,
            "Média demais": media_demais,
            "Mediana grupo P10": mediana_grupo,
            "Mediana demais": mediana_demais,
            "O grupo P10 fica (média)": "abaixo" if media_grupo < media_demais else "acima",
        }
    )

# DataFrame da pergunta 2.3: uma linha por variável
comparativo = pd.DataFrame(linhas_comparativo)
comparativo


### Como ler o comparativo

- Se o grupo P10 tem **menos população**, **menos indústria** e **mais administração pública**, isso descreve economias pequenas e dependentes do setor público.
- Alfabetização e IDHM um pouco **menores** reforçam um quadro social mais frágil.
- Isso **não** diz que “falta indústria causa PIB baixo” sozinha. Diz que, **em conjunto**, esses municípios compartilham um perfil parecido.

Abaixo: o mesmo grupo, município a município (DataFrame), e o gráfico da estrutura produtiva média.


In [ ]:
# Recorte das variáveis da pergunta 2.3, município a município (não é média)
grupo_p10_explicativas = grupo_p10[
    [
        "municipio",
        "pib_per_capita",
        "populacao_censo_2022",
        "densidade_demografica",
        "participacao_agropecuaria_pct",
        "participacao_industria_pct",
        "participacao_servicos_pct",
        "participacao_adm_publica_pct",
        "taxa_alfabetizacao_15_mais_pct",
        "idhm",
        "populacao_ocupada",
    ]
].copy()

display(grupo_p10_explicativas)

# DataFrame longo (tidy) para o gráfico de barras agrupadas
setores = pd.DataFrame(
    {
        "Setor": ["Agropecuária", "Indústria", "Serviços", "Adm. pública"] * 2,
        "Grupo": (
            ["Abaixo do P10"] * 4
            + ["Demais municípios"] * 4
        ),
        "% do VAB (2021)": [
            grupo_p10["participacao_agropecuaria_pct"].mean(),
            grupo_p10["participacao_industria_pct"].mean(),
            grupo_p10["participacao_servicos_pct"].mean(),
            grupo_p10["participacao_adm_publica_pct"].mean(),
            demais["participacao_agropecuaria_pct"].mean(),
            demais["participacao_industria_pct"].mean(),
            demais["participacao_servicos_pct"].mean(),
            demais["participacao_adm_publica_pct"].mean(),
        ],
    }
)

fig_setores = px.bar(
    setores,
    x="Setor",
    y="% do VAB (2021)",
    color="Grupo",
    barmode="group",
    title="Estrutura produtiva média: grupo P10 versus demais municípios (VAB 2021)",
    color_discrete_map={"Abaixo do P10": "#C00000", "Demais municípios": "#4F81BD"},
)
fig_setores.show()


---
# Pergunta 3. Descrição sumária da situação

Este parágrafo junta as respostas 1, 2.1, 2.2 e 2.3. Os números vêm das células anteriores, não foram digitados à mão.


In [ ]:
n_grupo = len(grupo_p10)
nomes = ", ".join(grupo_p10["municipio"].tolist())

# value_counts() conta quantos municípios do grupo há em cada mesorregião
meso = grupo_p10["mesorregiao"].value_counts().head(3)
meso_txt = "; ".join(f"{nome} ({qtd})" for nome, qtd in meso.items())

descricao = (
    f"O PIB per capita de 2023 dos {len(df)} municípios do Rio Grande do Norte "
    f"tem percentil 10 igual a R$ {br(percentil_10)}. Ficaram abaixo dessa separatriz "
    f"{n_grupo} municípios: {nomes}. "
    f"Nesse grupo, a média do PIB per capita é R$ {br(grupo_p10['pib_per_capita'].mean())}, "
    f"frente a R$ {br(df['pib_per_capita'].mean())} na média estadual. "
    f"São, em geral, municípios pequenos (população média de "
    f"{br(grupo_p10['populacao_censo_2022'].mean(), 0)} habitantes, contra "
    f"{br(demais['populacao_censo_2022'].mean(), 0)} nos demais) e de baixa densidade "
    f"({br(grupo_p10['densidade_demografica'].mean(), 1)} hab/km² contra "
    f"{br(demais['densidade_demografica'].mean(), 1)}). "
    f"A estrutura produtiva de 2021 indica maior dependência da administração pública "
    f"({br(grupo_p10['participacao_adm_publica_pct'].mean(), 1)}% do VAB no grupo P10 "
    f"contra {br(demais['participacao_adm_publica_pct'].mean(), 1)}% nos demais) e "
    f"menor peso da indústria ({br(grupo_p10['participacao_industria_pct'].mean(), 1)}% "
    f"contra {br(demais['participacao_industria_pct'].mean(), 1)}%). "
    f"Há também pior desempenho educacional e de desenvolvimento humano: taxa de "
    f"alfabetização de 15 anos ou mais de "
    f"{br(grupo_p10['taxa_alfabetizacao_15_mais_pct'].mean(), 1)}% "
    f"(contra {br(demais['taxa_alfabetizacao_15_mais_pct'].mean(), 1)}%) "
    f"e IDHM médio de {br(grupo_p10['idhm'].mean(), 3)} "
    f"(contra {br(demais['idhm'].mean(), 3)}). "
    f"A concentração territorial do grupo está principalmente em: {meso_txt}. "
    f"Em conjunto, o recorte aponta municípios de pequeno porte, com economia pouco "
    f"diversificada, forte peso do setor público e indicadores sociais mais frágeis, "
    f"o que ajuda a explicar o PIB per capita situado no décimo inferior da distribuição estadual."
)

print(descricao)


# Passo final. Atualizar o Excel para o Google Planilhas

O arquivo `data/municipios_rn.xlsx` passa a ter quatro abas, uma por bloco da atividade:

| Aba | Conteúdo |
| --- | --- |
| `Todos_municipios` | Pergunta 1 |
| `Grupo_P10` | Pergunta 2.2 |
| `Comparativo` | Pergunta 2.3 |
| `Resumo` | P10, quantidade no grupo e a descrição |

Importe de novo no Google Planilhas se você já tinha aberto a versão só com a primeira aba.


In [ ]:
resumo_aba = pd.DataFrame(
    [
        {"item": "Estado", "valor": "Rio Grande do Norte"},
        {"item": "Ano do PIB per capita", "valor": 2023},
        {"item": "Ano do VAB setorial", "valor": 2021},
        {"item": "Ano do Censo", "valor": 2022},
        {"item": "Percentil 10 (R$)", "valor": percentil_10},
        {"item": "Municípios abaixo do P10", "valor": len(grupo_p10)},
        {"item": "Total de municípios", "valor": len(df)},
        {"item": "Fórmula no Google Planilhas", "valor": "=PERCENTILE(coluna_pib; 0,1)"},
        {"item": "Descrição", "valor": descricao},
        {
            "item": "Fonte",
            "valor": "IBGE Cidades@ (pesquisa 38, indicador 47001) e APIs oficiais",
        },
    ]
)

with pd.ExcelWriter(arquivo_xlsx, engine="openpyxl") as excel:
    pib_todos.to_excel(excel, sheet_name="Todos_municipios", index=False)
    grupo_p10_resposta.to_excel(excel, sheet_name="Grupo_P10", index=False)
    comparativo.to_excel(excel, sheet_name="Comparativo", index=False)
    resumo_aba.to_excel(excel, sheet_name="Resumo", index=False)

print("Excel atualizado em:", arquivo_xlsx)
print("Abas: Todos_municipios, Grupo_P10, Comparativo, Resumo")
print("No Colab: baixe pela pasta à esquerda e abra no Google Planilhas (Arquivo → Importar).")


---
# Recado final

| Pergunta | O que este notebook fez |
| --- | --- |
| **Dados** | Baixou os indicadores do IBGE Cidades@ / APIs oficiais e montou um DataFrame |
| **Excel** | Exportou `.xlsx` para uso no Google Planilhas |
| **1** | Listou o PIB per capita dos 167 municípios (2023) |
| **2.1** | Calculou o percentil 10 (passo a passo + NumPy; equivalente ao PERCENTILE da planilha) |
| **2.2** | Filtrou quem tem PIB per capita **menor que** o P10 |
| **2.3** | Comparou população, estrutura produtiva, alfabetização e IDHM |
| **Descrição** | Juntou esses números em um parágrafo |

Tudo roda neste arquivo. Para repetir a coleta, execute de novo as células do passo 1.

URLs oficiais e dicionário das colunas: `README.md`.
